# Periodic spatial-ring weight — exact exponential gate

This fail-closed notebook runs one immutable judge in normal and optimised Python. It licenses only the exact exponential rewrite of `spatialWeightRing`; it does not license a square-root, weighted-kernel, Jordan–Wigner, or spectral theorem.

In [ ]:
import datetime, hashlib, json, os, platform, subprocess, sys, tempfile, urllib.request
from pathlib import Path

RAW_COMMIT = '8de79128957a1d6033e589c3c2f58eb53915416e'
RAW_URL = ('https://raw.githubusercontent.com/lluiseriksson/'
    f'THE-ERIKSSON-PROGRAMME/{RAW_COMMIT}/scripts/judge_spatial_ring_exponential.py')
EXPECTED_SHA256 = '16cc35d05e39ca334212da25fac8100d5914adade7a05d735659048366ecace6'
EXPECTED_CONFIGS = 510
EXPECTED_MUTATIONS = 510

run_root = Path(tempfile.mkdtemp(prefix='spatial-ring-exponential-'))
judge = run_root / 'judge_spatial_ring_exponential.py'
source = urllib.request.urlopen(RAW_URL, timeout=60).read()
source_hash = hashlib.sha256(source).hexdigest()
if source_hash != EXPECTED_SHA256:
    raise RuntimeError(f'judge SHA-256 mismatch: {source_hash}')
judge.write_bytes(source)

record = {
  'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'python': platform.python_version(),
  'cpu_count': os.cpu_count(), 'raw_commit': RAW_COMMIT,
  'judge_sha256': source_hash, 'runs': [],
}
for flags in ([], ['-O']):
    command = [sys.executable, *flags, str(judge)]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False)
    print('$', ' '.join(command))
    print(result.stdout, end='')
    print(f'[exit {result.returncode}]')
    if result.returncode != 0:
        raise RuntimeError(f'judge failed under flags {flags}')
    payload = json.loads(result.stdout)
    if payload.get('status') != 'PASS':
        raise RuntimeError(f'judge omitted PASS under flags {flags}')
    if payload.get('configurations_checked') != EXPECTED_CONFIGS:
        raise RuntimeError(f'wrong configuration count under flags {flags}: {payload}')
    if payload.get('closing_bond_mutations_rejected') != EXPECTED_MUTATIONS:
        raise RuntimeError(f'wrong mutation count under flags {flags}: {payload}')
    record['runs'].append({'flags': flags, 'exit': result.returncode, 'payload': payload})

artifact = run_root / 'spatial_ring_exponential_gate.json'
artifact.write_text(json.dumps(record, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(f'artifact_sha256={hashlib.sha256(artifact.read_bytes()).hexdigest()}')
print('SPATIAL RING-EXPONENTIAL GATE PASS')
from google.colab import files
files.download(str(artifact))